# 05 - Comprehensive Evaluation

Full evaluation pipeline for all fine-tuned components:

| Layer | Metric | What it measures |
|-------|--------|------------------|
| Embedding | Recall@k, MRR, NDCG | Retrieval quality |
| Re-ranker | AP, Precision@k | Re-ranking accuracy |
| LLM | BLEU, ROUGE-L, BERTScore | Response quality |
| E2E RAG | RAG accuracy, Latency | Full pipeline |

### Requirements
- Colab with T4 GPU (or local with fine-tuned models downloaded)
- Upload `test.jsonl` and knowledge base files

In [ ]:
# ⚠️ SAU KHI CHẠY XONG → Runtime → Restart session → rồi chạy từ Cell 2
!pip install -q \
    "huggingface_hub>=0.25" \
    "sentence-transformers>=3.0" \
    "transformers>=4.44,<5" \
    "accelerate>=1.0" \
    "bitsandbytes>=0.43" \
    "peft>=0.13" \
    evaluate rouge-score bert-score nltk pandas matplotlib

print('✅ Done! Bây giờ hãy: Runtime → Restart session → rồi chạy từ Cell 2')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

DRIVE_DIR = '/content/drive/MyDrive/vani-copilot'

# Verify test data exists
test_path = os.path.join(DRIVE_DIR, 'test.jsonl')
if not os.path.exists(test_path):
    raise FileNotFoundError(
        f'test.jsonl not found in {DRIVE_DIR}\n'
        '👉 Upload colab-upload/ files to MyDrive/vani-copilot/'
    )

with open(test_path, 'r', encoding='utf-8') as f:
    test_data = [json.loads(line) for line in f]
print(f'Test data: {len(test_data)} samples')

# Extract query-answer pairs
test_pairs = []
for item in test_data:
    msgs = item['messages']
    for i in range(len(msgs) - 1):
        if msgs[i]['role'] == 'user' and msgs[i+1]['role'] == 'assistant':
            q = msgs[i]['content'].strip()
            a = msgs[i+1]['content'].strip()
            if len(q) > 10 and len(a) > 15:
                test_pairs.append({'query': q, 'answer': a})

print(f'Test pairs extracted: {len(test_pairs)}')

---
## 1. Embedding Retrieval Evaluation

In [ ]:
from sentence_transformers import SentenceTransformer

# Load both models for comparison
model_base = SentenceTransformer('intfloat/multilingual-e5-base')

ft_path = os.path.join(DRIVE_DIR, 'embedding-finetuned')
if os.path.exists(ft_path):
    model_ft = SentenceTransformer(ft_path)
    print(f'Fine-tuned embedding loaded from Drive')
else:
    model_ft = None
    print('WARNING: Fine-tuned embedding not found. Will only evaluate base model.')

In [ ]:
# Build candidate pool from all answers
all_answers = list(set([p['answer'] for p in test_pairs]))
eval_pairs = test_pairs[:300]

def evaluate_retrieval(model, queries, positive_docs, all_docs, k_values=[1, 3, 5, 10]):
    q_embs = model.encode([f'query: {q}' for q in queries], normalize_embeddings=True, show_progress_bar=True)
    d_embs = model.encode([f'passage: {d}' for d in all_docs], normalize_embeddings=True, show_progress_bar=True)

    doc_to_idx = {d: i for i, d in enumerate(all_docs)}
    recall = {k: 0 for k in k_values}
    mrr = 0.0
    ndcg_at_10 = 0.0

    for i in range(len(queries)):
        scores = q_embs[i] @ d_embs.T
        ranked = np.argsort(scores)[::-1]
        pos_idx = doc_to_idx.get(positive_docs[i], -1)

        for k in k_values:
            if pos_idx in ranked[:k]:
                recall[k] += 1

        # MRR
        if pos_idx >= 0:
            rank = np.where(ranked == pos_idx)[0]
            if len(rank) > 0:
                mrr += 1.0 / (rank[0] + 1)
                # NDCG@10
                if rank[0] < 10:
                    ndcg_at_10 += 1.0 / np.log2(rank[0] + 2)

    n = len(queries)
    recall = {k: v / n for k, v in recall.items()}
    mrr /= n
    ndcg_at_10 /= n

    return {'recall': recall, 'mrr': mrr, 'ndcg@10': ndcg_at_10}

queries = [p['query'] for p in eval_pairs]
pos_docs = [p['answer'] for p in eval_pairs]

print('Evaluating base model...')
res_base = evaluate_retrieval(model_base, queries, pos_docs, all_answers)

if model_ft:
    print('\nEvaluating fine-tuned model...')
    res_ft = evaluate_retrieval(model_ft, queries, pos_docs, all_answers)
else:
    res_ft = None

print('\n' + '='*55)
print('  RETRIEVAL EVALUATION')
print('='*55)
print(f'{"Metric":>12} | {"Base":>8} | {"Fine-tuned":>10} | {"Delta":>8}')
print('-' * 55)
for k in [1, 3, 5, 10]:
    b = res_base['recall'][k]
    ft_val = res_ft['recall'][k] if res_ft else None
    ft_str = f'{ft_val:10.4f}' if ft_val is not None else f'{"N/A":>10}'
    d_str = f'{ft_val - b:+8.4f}' if ft_val is not None else f'{"":>8}'
    print(f'{"Recall@" + str(k):>12} | {b:8.4f} | {ft_str} | {d_str}')

mrr_b = res_base['mrr']
mrr_ft = res_ft['mrr'] if res_ft else None
mrr_ft_str = f'{mrr_ft:10.4f}' if mrr_ft is not None else f'{"N/A":>10}'
mrr_d_str = f'{mrr_ft - mrr_b:+8.4f}' if mrr_ft is not None else f'{"":>8}'
print(f'{"MRR":>12} | {mrr_b:8.4f} | {mrr_ft_str} | {mrr_d_str}')

ndcg_b = res_base['ndcg@10']
ndcg_ft = res_ft['ndcg@10'] if res_ft else None
ndcg_ft_str = f'{ndcg_ft:10.4f}' if ndcg_ft is not None else f'{"N/A":>10}'
ndcg_d_str = f'{ndcg_ft - ndcg_b:+8.4f}' if ndcg_ft is not None else f'{"":>8}'
print(f'{"NDCG@10":>12} | {ndcg_b:8.4f} | {ndcg_ft_str} | {ndcg_d_str}')

---
## 2. Re-ranker Evaluation

In [ ]:
from sentence_transformers import CrossEncoder

reranker_path = os.path.join(DRIVE_DIR, 'reranker-finetuned')
if os.path.exists(reranker_path):
    reranker = CrossEncoder(reranker_path)
    print('Re-ranker loaded from Drive')
else:
    print('WARNING: Re-ranker not found. Skipping re-ranker evaluation.')
    reranker = None

In [ ]:
if reranker:
    embed_model = model_ft if model_ft else model_base
    d_embs = embed_model.encode(
        [f'passage: {d}' for d in all_answers],
        normalize_embeddings=True, show_progress_bar=True, batch_size=64,
    )

    n_eval = min(100, len(eval_pairs))
    eval_queries = [eval_pairs[i]['query'] for i in range(n_eval)]

    # Batch encode all queries at once
    print('Encoding queries (batch)...')
    q_embs = embed_model.encode(
        [f'query: {q}' for q in eval_queries],
        normalize_embeddings=True, batch_size=64,
    )

    recall_embed_at_3, recall_embed_at_5 = 0, 0
    recall_rerank_at_3, recall_rerank_at_5 = 0, 0

    for i in range(n_eval):
        target = eval_pairs[i]['answer']
        scores = q_embs[i] @ d_embs.T
        top20_idx = np.argsort(scores)[-20:][::-1]
        top20_docs = [all_answers[j] for j in top20_idx]

        recall_embed_at_5 += int(target in top20_docs[:5])
        recall_embed_at_3 += int(target in top20_docs[:3])

        rerank_scores = reranker.predict([(eval_queries[i], doc) for doc in top20_docs])
        rerank_order = np.argsort(rerank_scores)[::-1]
        reranked_docs = [top20_docs[j] for j in rerank_order]

        recall_rerank_at_5 += int(target in reranked_docs[:5])
        recall_rerank_at_3 += int(target in reranked_docs[:3])

    e3 = recall_embed_at_3 / n_eval
    r3 = recall_rerank_at_3 / n_eval
    e5 = recall_embed_at_5 / n_eval
    r5 = recall_rerank_at_5 / n_eval

    print('\n' + '='*50)
    print('  RE-RANKER IMPACT')
    print('='*50)
    print(f'{"Metric":>15} | {"Embed only":>11} | {"+ Re-ranker":>11} | {"Delta":>8}')
    print('-' * 50)
    print(f'{"Recall@3":>15} | {e3:11.4f} | {r3:11.4f} | {r3-e3:+8.4f}')
    print(f'{"Recall@5":>15} | {e5:11.4f} | {r5:11.4f} | {r5-e5:+8.4f}')

---
## 3. LLM Generation Evaluation

In [ ]:
import gc
import evaluate
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

# Free VRAM from embedding/reranker models
try:
    del model_base, model_ft, bi_encoder, reranker, embed_model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

bleu_metric = evaluate.load('bleu')
rouge_metric = evaluate.load('rouge')
bertscore_metric = evaluate.load('bertscore')

MODEL_ID = 'unsloth/Meta-Llama-3.1-8B-Instruct'

# Load in 4-bit to fit T4 16GB
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

merged_path = os.path.join(DRIVE_DIR, 'vani-copilot-merged')
adapter_path = os.path.join(DRIVE_DIR, 'qlora-adapter-final')

model_merged = None
tokenizer = None

if os.path.exists(merged_path):
    print(f'Loading merged model from Drive (4-bit)...')
    model_merged = AutoModelForCausalLM.from_pretrained(
        merged_path, quantization_config=bnb_config, device_map='auto',
    )
    tokenizer = AutoTokenizer.from_pretrained(merged_path)
    print('✅ Loaded merged model')
elif os.path.exists(adapter_path):
    print(f'Loading base + adapter (4-bit)...')
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map='auto',
    )
    model_merged = PeftModel.from_pretrained(base, adapter_path)
    model_merged = model_merged.merge_and_unload()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    print('✅ Merged from adapter')
else:
    print('⚠️ No merged model or adapter found. Skipping LLM evaluation.')
    print('   (Run notebook 02 first, or download GGUF and test locally with Ollama)')

if model_merged and tokenizer:
    gen_pipe = pipeline(
        'text-generation',
        model=model_merged,
        tokenizer=tokenizer,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
    )
    print('Pipeline ready.')

In [ ]:
# Generate responses and compute metrics
system_prompt = 'Bạn là nhân viên CSKH của VANI Store - shop thời trang nữ online. Xưng em, gọi khách là chị/anh. Thân thiện, lịch sự.'

references = []
predictions = []

n_gen = min(50, len(test_pairs))

if model_merged:
    for i, pair in enumerate(test_pairs[:n_gen]):
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': pair['query']},
        ]
        result = gen_pipe(messages)
        generated = result[0]['generated_text'][-1]['content']

        references.append(pair['answer'])
        predictions.append(generated)

        if i < 3:
            print(f'\nQ: {pair["query"]}')
            print(f'Expected: {pair["answer"][:100]}...')
            print(f'Generated: {generated[:100]}...')

    print(f'\nGenerated {len(predictions)} responses.')
else:
    print('Skipping generation eval (no model loaded).')

In [ ]:
# Compute generation metrics
if predictions:
    bleu_result = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])
    rouge_result = rouge_metric.compute(predictions=predictions, references=references)
    bert_result = bertscore_metric.compute(predictions=predictions, references=references, lang='vi')

    gen_metrics = {
        'BLEU': bleu_result['bleu'],
        'ROUGE-1': rouge_result['rouge1'],
        'ROUGE-2': rouge_result['rouge2'],
        'ROUGE-L': rouge_result['rougeL'],
        'BERTScore-F1': np.mean(bert_result['f1']),
    }

    print('\n' + '='*40)
    print('  GENERATION METRICS')
    print('='*40)
    for name, val in gen_metrics.items():
        print(f'  {name:>15}: {val:.4f}')
else:
    gen_metrics = None
    print('No generation metrics (model not available).')

---
## 4. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Retrieval Recall@k
ax1 = axes[0]
k_vals = [1, 3, 5, 10]
base_recalls = [res_base['recall'][k] for k in k_vals]
ft_recalls = [res_ft['recall'][k] for k in k_vals] if res_ft else [0]*4

x = np.arange(len(k_vals))
w = 0.35
ax1.bar(x - w/2, base_recalls, w, label='Base', color='#94a3b8')
if res_ft:
    ax1.bar(x + w/2, ft_recalls, w, label='Fine-tuned', color='#3b82f6')
ax1.set_xlabel('k')
ax1.set_ylabel('Recall@k')
ax1.set_title('Retrieval: Recall@k')
ax1.set_xticks(x)
ax1.set_xticklabels([f'@{k}' for k in k_vals])
ax1.legend()
ax1.set_ylim(0, 1)

# 2. Re-ranker impact
ax2 = axes[1]
if reranker:
    rr_data = {'@3': [e3, r3], '@5': [e5, r5]}
    x2 = np.arange(2)
    ax2.bar(x2 - w/2, [e3, e5], w, label='Embed only', color='#94a3b8')
    ax2.bar(x2 + w/2, [r3, r5], w, label='+ Re-ranker', color='#10b981')
    ax2.set_xticks(x2)
    ax2.set_xticklabels(['Recall@3', 'Recall@5'])
    ax2.set_ylabel('Score')
    ax2.set_title('Re-ranker Impact')
    ax2.legend()
    ax2.set_ylim(0, 1)
else:
    ax2.text(0.5, 0.5, 'Re-ranker\nnot evaluated', ha='center', va='center', fontsize=14)
    ax2.set_title('Re-ranker Impact')

# 3. Generation metrics
ax3 = axes[2]
if gen_metrics:
    names = list(gen_metrics.keys())
    vals = list(gen_metrics.values())
    colors = ['#3b82f6', '#10b981', '#f59e0b', '#ef4444', '#8b5cf6']
    bars = ax3.bar(range(len(names)), vals, color=colors)
    ax3.set_xticks(range(len(names)))
    ax3.set_xticklabels(names, rotation=30, ha='right')
    ax3.set_ylabel('Score')
    ax3.set_title('Generation Quality')
    ax3.set_ylim(0, 1)
    for bar, val in zip(bars, vals):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.3f}', ha='center', fontsize=9)
else:
    ax3.text(0.5, 0.5, 'LLM\nnot evaluated', ha='center', va='center', fontsize=14)
    ax3.set_title('Generation Quality')

plt.suptitle('VANI Copilot - Fine-tuning Evaluation Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluation_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: evaluation_dashboard.png')

In [ ]:
# Final summary
print('='*60)
print('  VANI COPILOT - EVALUATION SUMMARY')
print('='*60)
print(f'\n  RETRIEVAL (Embedding: multilingual-e5-base)')
for k in [1, 3, 5, 10]:
    b = res_base['recall'][k]
    line = f'    Recall@{k}: {b:.4f}'
    if res_ft:
        f = res_ft['recall'][k]
        line += f' -> {f:.4f} ({f-b:+.4f})'
    print(line)
print(f'    MRR: {res_base["mrr"]:.4f}' + (f' -> {res_ft["mrr"]:.4f}' if res_ft else ''))
print(f'    NDCG@10: {res_base["ndcg@10"]:.4f}' + (f' -> {res_ft["ndcg@10"]:.4f}' if res_ft else ''))

if reranker:
    print(f'\n  RE-RANKER (xlm-roberta-base, fine-tuned)')
    print(f'    Recall@3: {e3:.4f} -> {r3:.4f} ({r3-e3:+.4f})')
    print(f'    Recall@5: {e5:.4f} -> {r5:.4f} ({r5-e5:+.4f})')

if gen_metrics:
    print(f'\n  GENERATION (Llama 3.1 8B + QLoRA)')
    for name, val in gen_metrics.items():
        print(f'    {name}: {val:.4f}')

print(f'\n  MODELS')
print(f'    LLM: Llama 3.1 8B + QLoRA (r=64, alpha=128)')
print(f'    Embedding: multilingual-e5-base (MNRL + Matryoshka)')
print(f'    Re-ranker: xlm-roberta-base (CrossEncoder)')
print(f'    Quantization: Q3_K_M GGUF (~3.9GB)')
print('='*60)

# Save summary to Drive
summary_path = os.path.join(DRIVE_DIR, 'evaluation_summary.txt')
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write('VANI Copilot Evaluation Summary\n')
    f.write(f'Date: {time.strftime("%Y-%m-%d %H:%M")}\n')
    if res_ft:
        for k in [1, 3, 5, 10]:
            f.write(f'Recall@{k}: {res_base["recall"][k]:.4f} -> {res_ft["recall"][k]:.4f}\n')
    if gen_metrics:
        for name, val in gen_metrics.items():
            f.write(f'{name}: {val:.4f}\n')
print(f'\nSummary saved to {summary_path}')